# Phase 2 — Model V2 (Enhanced Architecture)

**4 key upgrades over V1:**
1. **Graph Features** — PageRank + community detection on counterparty network (40%)
2. **F2 Threshold** — Recall-weighted decision boundary (20%)
3. **Hard Label Pruning** — Drop extreme red herrings, not just down-weight (15%)
4. **Two-Pass Temporal Windows** — 30d coarse → 3d fine for tight IoU (15%)

---

## 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from glob import glob
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, mean_absolute_error,
                             mean_squared_error, fbeta_score, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import xgboost as xgb
import warnings, time, os, gc
warnings.filterwarnings("ignore")

DATA_DIR = "IITD-Tryst-Hackathon/Phase 2"
t0 = time.time()

train = pd.read_csv("features_train_p2.csv")
test  = pd.read_csv("features_test_p2.csv")
labels = pd.read_parquet(f"{DATA_DIR}/train_labels.parquet")
labels["mule_flag_date"] = pd.to_datetime(labels["mule_flag_date"], errors="coerce")
accounts = pd.read_parquet(f"{DATA_DIR}/accounts.parquet")
for col in ["account_opening_date", "freeze_date"]:
    accounts[col] = pd.to_datetime(accounts[col], errors="coerce")

drop_cols = ["account_id", "is_mule", "first_large_ts", "open_date"]
feature_cols = [c for c in train.columns if c not in drop_cols]
for c in feature_cols:
    train[c] = pd.to_numeric(train[c], errors="coerce")
    test[c]  = pd.to_numeric(test[c], errors="coerce")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Mule rate: {train['is_mule'].mean():.4f} ({train['is_mule'].sum()} mules)")

Train: (96091, 71) | Test: (64062, 70)
Mule rate: 0.0279 (2683 mules)


## 1 — UPGRADE 1: Graph Features (PageRank + Community)

Build the counterparty transaction graph and compute network centrality
metrics. This goes beyond simple degree counts — PageRank captures
**influence propagation** through the transaction network.

In [2]:
print("=" * 60)
print("UPGRADE 1: Graph Features (PageRank + Community)")
print("=" * 60)

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    os.system("pip install networkx -q")
    import networkx as nx
    HAS_NX = True

# Build transaction graph from sampled parts (full graph would be too large)
parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))
print(f"Building counterparty graph from {min(len(parts), 100)} parts...")

# Collect edges: account_id <-> counterparty_id
edges = []
all_account_ids = set(train["account_id"]) | set(test["account_id"])

t_g = time.time()
for i, p in enumerate(parts[:100]):  # Sample 25% of parts
    df = pd.read_parquet(p, columns=["account_id", "counterparty_id", "amount"])
    df = df[df["counterparty_id"].notna()]
    # Only keep edges involving our accounts
    df = df[df["account_id"].isin(all_account_ids)]
    # Aggregate: (account, counterparty) -> total volume
    edge_agg = df.groupby(["account_id", "counterparty_id"])["amount"].agg(
        ["sum", "count"]).reset_index()
    edge_agg.columns = ["src", "dst", "total_volume", "txn_count"]
    edges.append(edge_agg)
    del df
    if (i+1) % 25 == 0:
        print(f"  [{i+1}/100] parts processed ({time.time()-t_g:.0f}s)")

edge_df = pd.concat(edges, ignore_index=True)
del edges; gc.collect()

# Aggregate across parts
edge_df = edge_df.groupby(["src", "dst"]).agg(
    total_volume=("total_volume", "sum"),
    txn_count=("txn_count", "sum")
).reset_index()

print(f"Edge DataFrame: {edge_df.shape[0]:,} unique edges ({time.time()-t_g:.0f}s)")

UPGRADE 1: Graph Features (PageRank + Community)
Building counterparty graph from 100 parts...
  [25/100] parts processed (12s)
  [50/100] parts processed (23s)
  [75/100] parts processed (35s)
  [100/100] parts processed (46s)
Edge DataFrame: 711,777 unique edges (47s)


In [3]:
# Build NetworkX graph
G = nx.Graph()
for _, row in edge_df.iterrows():
    G.add_edge(row["src"], row["dst"], weight=row["txn_count"])

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

# PageRank
print("Computing PageRank...")
t_pr = time.time()
pr = nx.pagerank(G, alpha=0.85, max_iter=50, tol=1e-4)
print(f"  PageRank done ({time.time()-t_pr:.0f}s)")

# Clustering coefficient (local)
print("Computing clustering coefficients...")
t_cc = time.time()
cc = nx.clustering(G)
print(f"  Clustering done ({time.time()-t_cc:.0f}s)")

# Community detection — Louvain (fast)
print("Computing communities...")
t_com = time.time()
try:
    communities = nx.community.louvain_communities(G, resolution=1.0, seed=42)
    # Map node -> community_id
    node_community = {}
    for cid, comm in enumerate(communities):
        for node in comm:
            node_community[node] = cid
    # Community size per node
    comm_sizes = {}
    for cid, comm in enumerate(communities):
        for node in comm:
            comm_sizes[node] = len(comm)
    print(f"  Louvain: {len(communities)} communities ({time.time()-t_com:.0f}s)")
except Exception as e:
    print(f"  Louvain failed ({e}), using connected components")
    node_community = {}
    comm_sizes = {}
    for cid, comp in enumerate(nx.connected_components(G)):
        for node in comp:
            node_community[node] = cid
            comm_sizes[node] = len(comp)

# Betweenness centrality (sampled for speed)
print("Computing betweenness centrality (sampled)...")
t_bc = time.time()
bc = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()), seed=42)
print(f"  Betweenness done ({time.time()-t_bc:.0f}s)")

# Build graph feature DataFrame
graph_features = pd.DataFrame({
    "account_id": list(all_account_ids)
})
graph_features["pagerank"] = graph_features["account_id"].map(pr).fillna(0)
graph_features["clustering_coeff"] = graph_features["account_id"].map(cc).fillna(0)
graph_features["community_size"] = graph_features["account_id"].map(comm_sizes).fillna(0)
graph_features["betweenness"] = graph_features["account_id"].map(bc).fillna(0)

# Ego-network density (for each account, density of its immediate neighbors)
ego_density = {}
for aid in all_account_ids:
    if G.has_node(aid):
        neighbors = list(G.neighbors(aid))
        if len(neighbors) >= 2:
            subg = G.subgraph(neighbors)
            ego_density[aid] = nx.density(subg)
        else:
            ego_density[aid] = 0
    else:
        ego_density[aid] = 0
graph_features["ego_density"] = graph_features["account_id"].map(ego_density)

print(f"\nGraph features: {graph_features.shape}")
print(graph_features.describe().round(4))

Graph: 141,533 nodes, 711,777 edges
Computing PageRank...
  PageRank done (3s)
Computing clustering coefficients...
  Clustering done (127s)
Computing communities...
  Louvain: 169 communities (53s)
Computing betweenness centrality (sampled)...
  Betweenness done (963s)

Graph features: (160153, 6)
          pagerank  clustering_coeff  community_size  betweenness  ego_density
count  160153.0000          160153.0     160153.0000  160153.0000     160153.0
mean        0.0000               0.0        550.9100       0.0000          0.0
std         0.0000               0.0       2106.2607       0.0000          0.0
min         0.0000               0.0          0.0000       0.0000          0.0
25%         0.0000               0.0          0.0000       0.0000          0.0
50%         0.0000               0.0          0.0000       0.0000          0.0
75%         0.0000               0.0        150.0000       0.0000          0.0
max         0.0002               0.0      12730.0000       0.0025   

In [4]:
# Merge graph features into train/test
train = train.merge(graph_features, on="account_id", how="left")
test  = test.merge(graph_features, on="account_id", how="left")

# Update feature_cols
new_graph_cols = ["pagerank", "clustering_coeff", "community_size", "betweenness", "ego_density"]
feature_cols = feature_cols + new_graph_cols
print(f"Features after graph: {len(feature_cols)} total (+{len(new_graph_cols)} graph features)")
del G, edge_df; gc.collect()

Features after graph: 72 total (+5 graph features)


572

## 2 — UPGRADE 3: Hard Label Pruning (Red Herrings)

Instead of soft sample weights, **completely drop** extreme red herrings
from training. Only use soft weights for ambiguous cases.

In [5]:
print("=" * 60)
print("UPGRADE 3: Hard Label Pruning")
print("=" * 60)

X_all = train[feature_cols].values
y_all = train["is_mule"].values

# Phase 1: Generate OOF predictions to find red herrings
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_screen = np.zeros(len(y_all))

for fold, (tr, val) in enumerate(skf.split(X_all, y_all)):
    m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
        scale_pos_weight=(y_all[tr]==0).sum()/max((y_all[tr]==1).sum(),1),
        random_state=42, verbosity=-1, n_jobs=-1)
    m.fit(X_all[tr], y_all[tr], eval_set=[(X_all[val], y_all[val])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_screen[val] = m.predict_proba(X_all[val])[:,1]

# Identify red herrings
extreme_fake_mule = (y_all == 1) & (oof_screen < 0.02)   # HARD DROP
suspect_fake_mule = (y_all == 1) & (oof_screen >= 0.02) & (oof_screen < 0.1)  # Soft weight
suspect_missed    = (y_all == 0) & (oof_screen > 0.8)     # Soft weight

print(f"Extreme red herrings (hard drop, p<0.02): {extreme_fake_mule.sum()}")
print(f"Ambiguous mule labels (soft weight, p<0.1): {suspect_fake_mule.sum()}")
print(f"Suspected missed mules (soft weight, p>0.8): {suspect_missed.sum()}")

# HARD PRUNE — remove extreme cases
keep_mask = ~extreme_fake_mule
X_clean = X_all[keep_mask]
y_clean = y_all[keep_mask]
train_ids_clean = train["account_id"].values[keep_mask]

# Soft weights for remaining ambiguous cases
sample_weights = np.ones(len(y_clean))
# Map suspect masks to the cleaned indices
suspect_fake_clean = suspect_fake_mule[keep_mask]
suspect_missed_clean = suspect_missed[keep_mask]
sample_weights[suspect_fake_clean] = 0.3
sample_weights[suspect_missed_clean] = 0.3

print(f"\nTraining set: {len(y_clean):,} → {len(y_all):,} (dropped {extreme_fake_mule.sum()}) ")
print(f"Mules remaining: {y_clean.sum()} / {y_all.sum()} original")
print(f"Baseline OOF AUC: {roc_auc_score(y_all, oof_screen):.4f}")

UPGRADE 3: Hard Label Pruning
Extreme red herrings (hard drop, p<0.02): 1
Ambiguous mule labels (soft weight, p<0.1): 83
Suspected missed mules (soft weight, p>0.8): 0

Training set: 96,090 → 96,091 (dropped 1) 
Mules remaining: 2682 / 2683 original
Baseline OOF AUC: 0.9850


## 3 — Model Training (Enhanced 4-Model Ensemble)

In [6]:
print("=" * 60)
print("TRAINING: Enhanced 4-Model Ensemble")
print("=" * 60)

X_test = test[feature_cols].values
spw = (y_clean==0).sum() / max((y_clean==1).sum(), 1)

# Tuned params
lgb_params = dict(n_estimators=1500, learning_rate=0.025, max_depth=9, num_leaves=180,
    subsample=0.85, colsample_bytree=0.75, min_child_samples=25,
    reg_alpha=0.3, reg_lambda=1.5, scale_pos_weight=spw,
    random_state=42, verbosity=-1, n_jobs=-1)

xgb_params = dict(n_estimators=1200, learning_rate=0.025, max_depth=8,
    subsample=0.85, colsample_bytree=0.7, min_child_weight=25,
    reg_alpha=0.3, reg_lambda=2.0, scale_pos_weight=spw,
    random_state=42, verbosity=0, tree_method="hist", n_jobs=-1, eval_metric="auc")

# Model 4: Extra Trees (diversity for ensemble)
from sklearn.ensemble import ExtraTreesClassifier

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(y_clean))
oof_xgb = np.zeros(len(y_clean))
oof_lr  = np.zeros(len(y_clean))
oof_et  = np.zeros(len(y_clean))
t_lgb = np.zeros(len(X_test))
t_xgb = np.zeros(len(X_test))
t_lr  = np.zeros(len(X_test))
t_et  = np.zeros(len(X_test))

for fold, (tr, val) in enumerate(skf.split(X_clean, y_clean)):
    Xtr, Xval = X_clean[tr], X_clean[val]
    ytr, yval = y_clean[tr], y_clean[val]
    wtr = sample_weights[tr]

    # LightGBM
    m1 = lgb.LGBMClassifier(**lgb_params)
    m1.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xval, yval)],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val] = m1.predict_proba(Xval)[:,1]
    t_lgb += m1.predict_proba(X_test)[:,1] / n_folds

    # XGBoost
    m2 = xgb.XGBClassifier(**xgb_params)
    m2.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xval, yval)], verbose=False)
    oof_xgb[val] = m2.predict_proba(Xval)[:,1]
    t_xgb += m2.predict_proba(X_test)[:,1] / n_folds

    # Logistic Regression
    lr = Pipeline([("imp", SimpleImputer(strategy="median")),
                   ("sc", StandardScaler()),
                   ("lr", LogisticRegression(C=0.1, class_weight="balanced",
                                              max_iter=1000, random_state=42))])
    lr.fit(Xtr, ytr)
    oof_lr[val] = lr.predict_proba(Xval)[:,1]
    t_lr += lr.predict_proba(X_test)[:,1] / n_folds

    # Extra Trees (4th model for diversity)
    imp = SimpleImputer(strategy="median")
    Xtr_imp = imp.fit_transform(Xtr)
    Xval_imp = imp.transform(Xval)
    Xtest_imp = imp.transform(X_test)
    m4 = ExtraTreesClassifier(n_estimators=500, max_depth=12, min_samples_leaf=20,
                               class_weight="balanced", random_state=42, n_jobs=-1)
    m4.fit(Xtr_imp, ytr, sample_weight=wtr)
    oof_et[val] = m4.predict_proba(Xval_imp)[:,1]
    t_et += m4.predict_proba(Xtest_imp)[:,1] / n_folds

    auc_lgb = roc_auc_score(yval, oof_lgb[val])
    auc_xgb = roc_auc_score(yval, oof_xgb[val])
    print(f"  Fold {fold+1}: LGB={auc_lgb:.4f} XGB={auc_xgb:.4f}")

print(f"\nOOF AUC:")
print(f"  LightGBM:    {roc_auc_score(y_clean, oof_lgb):.4f}")
print(f"  XGBoost:     {roc_auc_score(y_clean, oof_xgb):.4f}")
print(f"  LogReg:      {roc_auc_score(y_clean, oof_lr):.4f}")
print(f"  ExtraTrees:  {roc_auc_score(y_clean, oof_et):.4f}")

# Ensemble: LGB 40% + XGB 30% + LR 15% + ET 15%
oof_ens = 0.40*oof_lgb + 0.30*oof_xgb + 0.15*oof_lr + 0.15*oof_et
t_ens   = 0.40*t_lgb   + 0.30*t_xgb   + 0.15*t_lr   + 0.15*t_et
print(f"  ENSEMBLE:    {roc_auc_score(y_clean, oof_ens):.4f}")

TRAINING: Enhanced 4-Model Ensemble
  Fold 1: LGB=0.9889 XGB=0.9887
  Fold 2: LGB=0.9880 XGB=0.9879
  Fold 3: LGB=0.9883 XGB=0.9882
  Fold 4: LGB=0.9889 XGB=0.9892
  Fold 5: LGB=0.9879 XGB=0.9881

OOF AUC:
  LightGBM:    0.9884
  XGBoost:     0.9884
  LogReg:      0.9764
  ExtraTrees:  0.9713
  ENSEMBLE:    0.9847


## 4 — UPGRADE 2: F2 Threshold Optimization (Recall-Heavy)

In [7]:
print("=" * 60)
print("UPGRADE 2: F2 Threshold Optimization")
print("=" * 60)

# Search for optimal threshold that maximizes F2 (recall-weighted)
thresholds = np.arange(0.1, 0.95, 0.005)
best_f2 = 0
best_t_f2 = 0.5
best_f1 = 0
best_t_f1 = 0.5

for t in thresholds:
    preds = (oof_ens > t).astype(int)
    f2 = fbeta_score(y_clean, preds, beta=2)
    f1 = f1_score(y_clean, preds)
    if f2 > best_f2:
        best_f2 = f2
        best_t_f2 = t
    if f1 > best_f1:
        best_f1 = f1
        best_t_f1 = t

print(f"F1-optimal threshold: {best_t_f1:.3f} → F1={best_f1:.4f}")
print(f"F2-optimal threshold: {best_t_f2:.3f} → F2={best_f2:.4f}")

# Use F2 threshold for submission (recall-heavy)
preds_f2 = (oof_ens > best_t_f2).astype(int)
preds_f1 = (oof_ens > best_t_f1).astype(int)

print(f"\nF1 threshold metrics:")
print(f"  Precision: {precision_score(y_clean, preds_f1):.4f}")
print(f"  Recall:    {recall_score(y_clean, preds_f1):.4f}")
print(f"  F1:        {f1_score(y_clean, preds_f1):.4f}")

print(f"\nF2 threshold metrics:")
print(f"  Precision: {precision_score(y_clean, preds_f2):.4f}")
print(f"  Recall:    {recall_score(y_clean, preds_f2):.4f}")
print(f"  F1:        {f1_score(y_clean, preds_f2):.4f}")
print(f"  F2:        {fbeta_score(y_clean, preds_f2, beta=2):.4f}")
print(f"  FN saved:  {preds_f1.sum() - preds_f2.sum():+d} more mules caught")

UPGRADE 2: F2 Threshold Optimization
F1-optimal threshold: 0.780 → F1=0.8662
F2-optimal threshold: 0.505 → F2=0.8625

F1 threshold metrics:
  Precision: 0.9039
  Recall:    0.8315
  F1:        0.8662

F2 threshold metrics:
  Precision: 0.8117
  Recall:    0.8762
  F1:        0.8427
  F2:        0.8625
  FN saved:  -428 more mules caught


## 5 — Full Metrics

In [8]:
print("=" * 60)
print("FULL METRICS (OOF on cleaned training set)")
print("=" * 60)

preds = preds_f2  # Use F2-optimal threshold
cm = confusion_matrix(y_clean, preds)
auc = roc_auc_score(y_clean, oof_ens)
mae = mean_absolute_error(y_clean, oof_ens)
mse = mean_squared_error(y_clean, oof_ens)
rmse = np.sqrt(mse)
rmsle = np.sqrt(np.mean((np.log1p(np.clip(oof_ens,1e-8,1)) - np.log1p(np.clip(y_clean,1e-8,1)))**2))
smape = np.mean(2*np.abs(oof_ens-y_clean)/(np.abs(oof_ens)+np.abs(y_clean)+1e-8))*100

print(f"\n{'CLASSIFICATION'}")
print(f"{'─'*40}")
print(f"{'AUC-ROC':<20} {auc:.4f}")
print(f"{'F1-Score':<20} {f1_score(y_clean, preds):.4f}")
print(f"{'F2-Score':<20} {fbeta_score(y_clean, preds, beta=2):.4f}")
print(f"{'Accuracy':<20} {accuracy_score(y_clean, preds):.4f}")
print(f"{'Precision':<20} {precision_score(y_clean, preds):.4f}")
print(f"{'Recall':<20} {recall_score(y_clean, preds):.4f}")
print(f"\n{'REGRESSION (prob vs label)'}")
print(f"{'─'*40}")
print(f"{'MAE':<20} {mae:.4f}")
print(f"{'MSE':<20} {mse:.4f}")
print(f"{'RMSE':<20} {rmse:.4f}")
print(f"{'RMSLE':<20} {rmsle:.4f}")
print(f"{'SMAPE':<20} {smape:.2f}%")
print(f"\n{'CONFUSION MATRIX'}")
print(f"{'─'*40}")
print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")

FULL METRICS (OOF on cleaned training set)

CLASSIFICATION
────────────────────────────────────────
AUC-ROC              0.9847
F1-Score             0.8427
F2-Score             0.8625
Accuracy             0.9909
Precision            0.8117
Recall               0.8762

REGRESSION (prob vs label)
────────────────────────────────────────
MAE                  0.0512
MSE                  0.0111
RMSE                 0.1054
RMSLE                0.0845
SMAPE                195.06%

CONFUSION MATRIX
────────────────────────────────────────
  TN=92,863  FP=545
  FN=332  TP=2,350


## 6 — UPGRADE 4: Two-Pass Temporal Windows (Tight IoU)

**Pass 1:** 30-day rolling z-score → identify the *active month*
**Pass 2:** 3-day rolling z-score within that month → pinpoint exact start/end

In [9]:
print("=" * 60)
print("UPGRADE 4: Two-Pass Temporal Windows")
print("=" * 60)

test_probs = t_ens.copy()
test_ids = test["account_id"].values

# Threshold for temporal analysis
mule_threshold = best_t_f2 * 0.5  # Analyze accounts at half the F2 threshold
high_prob_ids = set(test_ids[test_probs > mule_threshold])
print(f"Accounts needing temporal windows: {len(high_prob_ids):,}")

# Build daily volume series
daily_series = {}
parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))

for i, p in enumerate(parts):
    df = pd.read_parquet(p, columns=["account_id", "transaction_timestamp", "amount"])
    df = df[df["account_id"].isin(high_prob_ids)]
    if len(df) == 0: continue
    df["ts"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    df["date"] = df["ts"].dt.date
    df["abs_amount"] = df["amount"].abs()
    dv = df.groupby(["account_id", "date"])["abs_amount"].sum()
    for (aid, date), vol in dv.items():
        if aid not in daily_series:
            daily_series[aid] = {}
        daily_series[aid][date] = daily_series[aid].get(date, 0) + vol
    del df
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(parts)}] processed")
    gc.collect()

print(f"Built daily series for {len(daily_series):,} accounts")

UPGRADE 4: Two-Pass Temporal Windows
Accounts needing temporal windows: 1,857
  [100/396] processed
  [200/396] processed
  [300/396] processed
Built daily series for 1,609 accounts


In [10]:
from datetime import timedelta, datetime

def two_pass_temporal_window(daily_vol_dict):
    """Two-pass: coarse (30d) → fine (3d) temporal detection."""
    if len(daily_vol_dict) < 10:
        return None, None

    dates = sorted(daily_vol_dict.keys())
    vols = np.array([daily_vol_dict[d] for d in dates])

    # ── PASS 1: 30-day rolling z-score → find active month ──
    lookback = 30
    z_scores = np.zeros(len(vols))
    for i in range(lookback, len(vols)):
        baseline = vols[max(0, i - lookback):i]
        mu, sigma = baseline.mean(), baseline.std()
        if sigma > 0:
            z_scores[i] = (vols[i] - mu) / sigma

    # Find the densest anomalous period (30-day window with most z>2 days)
    coarse_flags = z_scores > 2.0
    if coarse_flags.sum() == 0:
        # Fallback: top 5% volume days
        vol_thresh = np.percentile(vols, 95)
        coarse_flags = vols >= vol_thresh

    if coarse_flags.sum() == 0:
        return None, None

    # Find the active month (densest 30-day window)
    flagged_idx = np.where(coarse_flags)[0]
    best_window_start = flagged_idx[0]
    best_window_end = flagged_idx[-1]

    # ── PASS 2: 3-day rolling z-score within active period ──
    # Expand the window slightly for context
    expand = 15
    fine_start = max(0, best_window_start - expand)
    fine_end = min(len(vols), best_window_end + expand)

    fine_vols = vols[fine_start:fine_end]
    fine_dates = [dates[i] for i in range(fine_start, fine_end)]

    if len(fine_vols) < 5:
        # Use coarse boundaries
        start = dates[best_window_start]
        end = dates[best_window_end]
        return f"{start}T00:00:00", f"{end}T23:59:59"

    # Fine z-scores with 3-day lookback
    fine_z = np.zeros(len(fine_vols))
    fine_lookback = 3
    for i in range(fine_lookback, len(fine_vols)):
        baseline = fine_vols[max(0, i - fine_lookback):i]
        mu, sigma = baseline.mean(), baseline.std()
        if sigma > 0:
            fine_z[i] = (fine_vols[i] - mu) / sigma

    # Fine anomalous days (lower threshold for precision)
    fine_flags = fine_z > 1.5
    if fine_flags.sum() == 0:
        fine_flags = fine_vols >= np.percentile(fine_vols, 80)

    fine_flagged = np.where(fine_flags)[0]
    if len(fine_flagged) == 0:
        start = dates[best_window_start]
        end = dates[best_window_end]
    else:
        start = fine_dates[fine_flagged[0]]
        end = fine_dates[fine_flagged[-1]]

    return f"{start}T00:00:00", f"{end}T23:59:59"

# Apply two-pass detection
temporal_windows = {}
for aid in high_prob_ids:
    if aid in daily_series:
        start, end = two_pass_temporal_window(daily_series[aid])
        temporal_windows[aid] = (start, end)
    else:
        temporal_windows[aid] = (None, None)

n_with_window = sum(1 for s, e in temporal_windows.values() if s is not None)
print(f"Accounts with fine-grained windows: {n_with_window:,}/{len(high_prob_ids):,}")

# Show window width stats
widths = []
for s, e in temporal_windows.values():
    if s and e:
        sd = pd.to_datetime(s)
        ed = pd.to_datetime(e)
        widths.append((ed - sd).days)
if widths:
    wa = np.array(widths)
    print(f"Window width stats: median={np.median(wa):.0f}d, mean={wa.mean():.0f}d, "
          f"p25={np.percentile(wa,25):.0f}d, p75={np.percentile(wa,75):.0f}d")

Accounts with fine-grained windows: 1,601/1,857
Window width stats: median=595d, mean=672d, p25=296d, p75=828d


## 7 — Generate Submission

In [11]:
print("=" * 60)
print("GENERATING SUBMISSION V2")
print("=" * 60)

submission = pd.DataFrame({
    "account_id": test_ids,
    "is_mule": test_probs,
    "suspicious_start": "",
    "suspicious_end": ""
})

for idx in range(len(submission)):
    aid = submission.iloc[idx]["account_id"]
    if aid in temporal_windows:
        start, end = temporal_windows[aid]
        if start:
            submission.iat[idx, 2] = start
            submission.iat[idx, 3] = end

submission.to_csv("submission_v2.csv", index=False)

print(f"Submission: {submission.shape}")
print(f"  Mean prob:       {submission['is_mule'].mean():.4f}")
print(f"  >50% mule:       {(submission['is_mule']>0.5).sum():,}")
print(f"  >80% mule:       {(submission['is_mule']>0.8).sum():,}")
print(f"  With windows:    {(submission['suspicious_start']!='').sum():,}")
print(f"\n✅ submission_v2.csv saved")
print(f"Total time: {time.time()-t0:.0f}s = {(time.time()-t0)/60:.0f} min")

GENERATING SUBMISSION V2
Submission: (64062, 4)
  Mean prob:       0.0577
  >50% mule:       848
  >80% mule:       525
  With windows:    1,601

✅ submission_v2.csv saved
Total time: 2225s = 37 min


## 8 — V1 vs V2 Comparison

In [12]:
print("=" * 60)
print("V1 vs V2 COMPARISON")
print("=" * 60)
print(f"""
{'Dimension':<30} {'V1':>12} {'V2':>12}
{'─'*56}
{'Architecture':<30} {'3-model':>12} {'4-model+graph':>12}
{'Graph Features':<30} {'None':>12} {'PageRank+5':>12}
{'Red Herring Handling':<30} {'Soft weight':>12} {'Hard prune':>12}
{'Threshold Strategy':<30} {'F1-optimal':>12} {'F2-optimal':>12}
{'Temporal Windows':<30} {'1-pass 30d':>12} {'2-pass 30d+3d':>12}
{'AUC-ROC':<30} {'0.9867':>12} {roc_auc_score(y_clean, oof_ens):>12.4f}
{'F1':<30} {'0.8657':>12} {f1_score(y_clean, preds_f1):>12.4f}
{'F2':<30} {'N/A':>12} {fbeta_score(y_clean, preds_f2, beta=2):>12.4f}
{'Recall':<30} {'0.836':>12} {recall_score(y_clean, preds_f2):>12.4f}
""")

# Feature importance (last fold LGB)
imp = pd.DataFrame({"feature": feature_cols, "importance": m1.feature_importances_})
imp = imp.sort_values("importance", ascending=False)
print("Top 20 features (including graph):")
for _, r in imp.head(20).iterrows():
    marker = " ★" if r["feature"] in new_graph_cols else ""
    print(f"  {r['feature']:<35} {r['importance']:>6.0f}{marker}")

V1 vs V2 COMPARISON

Dimension                                V1           V2
────────────────────────────────────────────────────────
Architecture                        3-model 4-model+graph
Graph Features                         None   PageRank+5
Red Herring Handling            Soft weight   Hard prune
Threshold Strategy               F1-optimal   F2-optimal
Temporal Windows                 1-pass 30d 2-pass 30d+3d
AUC-ROC                              0.9867       0.9847
F1                                   0.8657       0.8662
F2                                      N/A       0.8625
Recall                                0.836       0.8762

Top 20 features (including graph):
  median_dwell_hours                    2710
  composite_score                       2660
  gap_std                               2494
  branch_mule_rate                      2363
  fanin_fanout_ratio                    2360
  gap_mean                              2235
  life_ratio                            2201